<a href="https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/course-2-generative-ai/lab-12-shrink-the-cobalt-assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 12 (graded) — Shrink the Cobalt assistant
**Course 2: Generative AI and LLMs with Python — Chapter 12: Inference systems**

**Problem brief (Leo Farkas, Cobalt Manufacturing):** "We want the maintenance assistant
running on our own hardware, offline, on the plant floor. It's too slow and too big."

**What you'll submit:** a quantized model with measured size/latency/quality, a small
`vllm` batching demo (best-effort — heavy to install, a documented skip is acceptable), a
CPU `llama.cpp` inference demo (the offline fallback), and a self-host-vs-managed recommendation.

In [ ]:
!pip install -q transformers bitsandbytes accelerate llama-cpp-python huggingface_hub

## 1. Baseline: full-precision size and latency

In [ ]:
import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

def model_size_mb(model):
    return sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2

def measure_latency(model, prompt, n_runs=5, max_new_tokens=40):
    ids = tokenizer(prompt, return_tensors='pt').input_ids.to(model.device)
    with torch.no_grad():
        model.generate(ids, max_new_tokens=5, pad_token_id=tokenizer.eos_token_id)  # warm-up
        t0 = time.perf_counter()
        for _ in range(n_runs):
            model.generate(ids, max_new_tokens=max_new_tokens, pad_token_id=tokenizer.eos_token_id)
        return (time.perf_counter() - t0) / n_runs

fp_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32).to('cpu')
prompt = 'The maintenance procedure for a worn bearing is'
fp_size = model_size_mb(fp_model)
fp_latency = measure_latency(fp_model, prompt)
print(f'Full precision: {fp_size:.0f} MB, {fp_latency*1000:.0f} ms/40 tokens on CPU')

## 2. 4-bit quantization (bitsandbytes, GPU-only)

In [ ]:
from transformers import BitsAndBytesConfig

if torch.cuda.is_available():
    bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    q_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
    q_size = model_size_mb(q_model)  # note: reports the in-memory 4-bit footprint
    q_latency = measure_latency(q_model, prompt)
    print(f'4-bit (bitsandbytes): ~{q_size:.0f} MB, {q_latency*1000:.0f} ms/40 tokens on GPU')
    print(f'Size reduction vs. full precision: {(1 - q_size/fp_size):.0%}')
else:
    print('No GPU available — bitsandbytes 4-bit inference needs one. Skipping this section;')
    print('the GGUF/llama.cpp path below is the CPU-appropriate quantization path anyway.')

## 3. GGUF + `llama.cpp`: the CPU / edge path (and the offline fallback)
Converting a model to GGUF from scratch needs the `llama.cpp` repo's convert script; this
notebook instead pulls an already-quantized GGUF build from the Hub — the realistic path for
most teams, and the one that actually demonstrates CPU inference speed and quality.

In [ ]:
def load_gguf_model():
    try:
        from llama_cpp import Llama
        from huggingface_hub import hf_hub_download
        path = hf_hub_download(repo_id='Qwen/Qwen2.5-0.5B-Instruct-GGUF', filename='qwen2.5-0.5b-instruct-q4_k_m.gguf')
        llm = Llama(model_path=path, n_ctx=512, verbose=False)
        print('Loaded a real Q4_K_M GGUF quantized model.')
        return llm
    except Exception as e:
        print(f'GGUF download/load skipped ({e}) — this is expected without network access to')
        print('the Hub; the size/latency numbers above already demonstrate the quantization idea.')
        return None

gguf_model = load_gguf_model()
if gguf_model is not None:
    t0 = time.perf_counter()
    out = gguf_model(prompt, max_tokens=40)
    gguf_latency = time.perf_counter() - t0
    print(f'GGUF (Q4_K_M) CPU latency: {gguf_latency*1000:.0f} ms/40 tokens')
    print('Output:', out['choices'][0]['text'])

## 4. A tiny `vllm` batching demo (best-effort)

In [ ]:
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'vllm'], check=True, timeout=300)
    from vllm import LLM, SamplingParams
    vllm_model = LLM(model=MODEL_ID, gpu_memory_utilization=0.5)
    sampling = SamplingParams(max_tokens=40)
    prompts_batch = [prompt] * 8  # continuous batching handles this efficiently
    t0 = time.perf_counter()
    outputs = vllm_model.generate(prompts_batch, sampling)
    print(f'vLLM: {len(prompts_batch)} requests in {time.perf_counter() - t0:.1f}s (batched)')
except Exception as e:
    print(f'vLLM demo skipped ({e}) — vLLM needs a GPU + a longer install than this environment')
    print('allows right now. Read Part B6 of REPOS.md for the PagedAttention/continuous-batching')
    print('mechanism this would demonstrate; this skip does not block the lab.')

## 5. Self-host vs. managed recommendation (fill in)
Given Cobalt's actual requirement (offline, on-plant, no cloud dependency), which serving
path from this notebook fits, and why does that make the vLLM/managed-endpoint question
moot for this specific client even though it's the right answer for higher-volume, always-
connected use cases?

_Your answer here._

---
*Beacon AI · AIBits Academy — Chapter 12: Inference systems*